## Handle duplicate records
- dropDuplicate() --> function is used to remove duplicate rows from DataFrame
- you can either drop duplicates from all columns or specify a subset of columns based on which duplicate should be removed 


In [0]:
path = '/Volumes/student_demo/standard/input_data/employees.csv'

df = spark.read.format('csv').options(header='true',inferSchema='true').load(path)

# res = df.dropDuplicates()
res = df.dropDuplicates(['FIRST_NAME','LAST_NAME','JOB_ID'])
display(res)
# --205,200

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
202,Pat,null,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20
119,Karen,Colmenares,KCOLMENA,515.127.4566,10-Aug-07,PU_CLERK,2500,-,114,30
115,Alexander,Khoo,AKHOO,515.127.4562,18-May-03,PU_CLERK,3100,-,114,30
199,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,50
204,Hermann,Baer,HBAER,515.123.8888,null,PR_REP,10000,-,101,70
131,James,Marlow,JAMRLOW,650.124.7234,16-Feb-05,ST_CLERK,2500,-,121,50
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20
100,Steven,null,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90
101,Neena,Kochhar,NKOCHHAR,515.123.4568,21-Sep-05,AD_VP,17000,-,100,90
117,Sigal,Tobias,STOBIAS,515.127.4564,24-Jul-05,PU_CLERK,2800,-,114,30


In [0]:
df.createOrReplaceTempView('employ')

In [0]:
# To remove duplicate rows in SQL (a process known as deduplication)
final_df = spark.sql("""select * from (
select *,row_number()over(partition by FIRST_NAME,LAST_NAME,JOB_ID order by EMPLOYEE_ID) as rn from employ
) 
where rn=1
""").drop('rn')
display(final_df)

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
121,Adam,Fripp,AFRIPP,650.123.2234,10-Apr-05,ST_MAN,8200,-,100,50
103,Alexander,Hunold,AHUNOLD,590.423.4567,03-Jan-06,IT_PROG,9000,-,102,60
115,Alexander,Khoo,AKHOO,515.127.4562,18-May-03,PU_CLERK,3100,-,114,30
104,Bruce,Ernst,BERNST,590.423.4568,21-May-07,IT_PROG,6000,-,103,60
109,Daniel,Faviet,DFAVIET,515.124.4169,16-Aug-02,FI_ACCOUNT,null,-,108,100
105,David,Austin,DAUSTIN,590.423.4569,25-Jun-05,IT_PROG,4800,-,103,60
113,Den,Popp,LPOPP,515.124.4567,07-Dec-07,FI_ACCOUNT,6900,-,108,100
114,Den,Popp,DRAPHEAL,515.127.4561,07-Dec-02,PU_MAN,11000,-,100,30
107,Diana,Lorentz,DLORENTZ,590.423.5567,07-Feb-07,IT_PROG,4200,-,103,60
198,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,50


In [0]:
from pyspark.sql.functions import *
emp_final_res = final_df.withColumn('HIRE_DATE',to_date(col('HIRE_DATE'),'dd-MMM-yy')).\
    withColumn('FULL_NAME',expr("FIRST_NAME ||' '|| LAST_NAME")).\
    withColumn('employee_status',expr('''case when HIRE_DATE>='2008-01-01' then 'Recent Employee'
                                      when HIRE_DATE<'2008-01-01' then 'OLD Employee'
                                      else 'Unknown' end
                                      '''))
display(emp_final_res)

## if HIRE_DATE>='2008-01-01' then 'recent employ
## if HIRE_DATE<'2008-01-01' then 'OLD employ


EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,FULL_NAME,employee_status
121,Adam,Fripp,AFRIPP,650.123.2234,2005-04-10,ST_MAN,8200,-,100,50,Adam Fripp,OLD Employee
103,Alexander,Hunold,AHUNOLD,590.423.4567,2006-01-03,IT_PROG,9000,-,102,60,Alexander Hunold,OLD Employee
115,Alexander,Khoo,AKHOO,515.127.4562,2003-05-18,PU_CLERK,3100,-,114,30,Alexander Khoo,OLD Employee
104,Bruce,Ernst,BERNST,590.423.4568,2007-05-21,IT_PROG,6000,-,103,60,Bruce Ernst,OLD Employee
109,Daniel,Faviet,DFAVIET,515.124.4169,2002-08-16,FI_ACCOUNT,null,-,108,100,Daniel Faviet,OLD Employee
105,David,Austin,DAUSTIN,590.423.4569,2005-06-25,IT_PROG,4800,-,103,60,David Austin,OLD Employee
113,Den,Popp,LPOPP,515.124.4567,2007-12-07,FI_ACCOUNT,6900,-,108,100,Den Popp,OLD Employee
114,Den,Popp,DRAPHEAL,515.127.4561,2002-12-07,PU_MAN,11000,-,100,30,Den Popp,OLD Employee
107,Diana,Lorentz,DLORENTZ,590.423.5567,2007-02-07,IT_PROG,4200,-,103,60,Diana Lorentz,OLD Employee
198,Donald,OConnell,DOCONNEL,650.507.9833,2007-06-21,SH_CLERK,2600,-,124,50,Donald OConnell,OLD Employee


In [0]:
path = '/Volumes/student_demo/standard/input_data/employees.csv'

spark_df = spark.read.format('csv').options(header='true',inferSchema='true').load(path)

# Convert Spark DataFrame to Pandas dataframe
pandas_df = spark_df.toPandas()


,EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
0,198,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600.0,-,124,50
1,199,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600.0,-,124,50
2,200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400.0,-,101,10
3,200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400.0,-,101,10
4,201,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000.0,-,100,20
5,202,Pat,None,PFAY,603.123.6666,17-Aug-05,MK_REP,6000.0,-,201,20
6,203,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,NaN,-,101,40
7,205,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008.0,-,101,110
8,204,Hermann,Baer,HBAER,515.123.8888,None,PR_REP,10000.0,-,101,70
9,205,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008.0,-,101,110


In [0]:
# Convert Pandas dataframe to Spark DataFrame
spark_df2 = spark.createDataFrame(pandas_df)
display(spark_df2)

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
198,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,50
199,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,50
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20
202,Pat,null,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20
203,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,null,-,101,40
205,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110
204,Hermann,Baer,HBAER,515.123.8888,null,PR_REP,10000,-,101,70
205,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110


In [0]:
# Get distinct values from a specific column
distinct_values_df = spark_df2.select(col('JOB_ID')).distinct()
display(distinct_values_df)

JOB_ID
SH_CLERK
AD_ASST
MK_MAN
MK_REP
HR_REP
AC_MGR
PR_REP
AC_ACCOUNT
AD_PRES
AD_VP


In [0]:
dept_data = [
    (10, 'IT'),
    (20, 'Sales'),
    (30, 'Production'),
    (40, 'Chemical'),
    (50, 'Operation'),
    (60, 'Finance'),
    (70, 'HR'),
    (90, 'Marketing'),
    (100, 'R_and_D'),
    (110, 'Customer Service'),
    (120, 'Quality'),
    (130, 'Admin')
]

dept_df = spark.createDataFrame(dept_data, ['ID', 'Department_name'])
display(dept_df)

ID,Department_name
10,IT
20,Sales
30,Production
40,Chemical
50,Operation
60,Finance
70,HR
90,Marketing
100,R_and_D
110,Customer Service


In [0]:
display(emp_final_res)

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,FULL_NAME,employee_status
121,Adam,Fripp,AFRIPP,650.123.2234,2005-04-10,ST_MAN,8200,-,100,50,Adam Fripp,OLD Employee
103,Alexander,Hunold,AHUNOLD,590.423.4567,2006-01-03,IT_PROG,9000,-,102,60,Alexander Hunold,OLD Employee
115,Alexander,Khoo,AKHOO,515.127.4562,2003-05-18,PU_CLERK,3100,-,114,30,Alexander Khoo,OLD Employee
104,Bruce,Ernst,BERNST,590.423.4568,2007-05-21,IT_PROG,6000,-,103,60,Bruce Ernst,OLD Employee
109,Daniel,Faviet,DFAVIET,515.124.4169,2002-08-16,FI_ACCOUNT,null,-,108,100,Daniel Faviet,OLD Employee
105,David,Austin,DAUSTIN,590.423.4569,2005-06-25,IT_PROG,4800,-,103,60,David Austin,OLD Employee
113,Den,Popp,LPOPP,515.124.4567,2007-12-07,FI_ACCOUNT,6900,-,108,100,Den Popp,OLD Employee
114,Den,Popp,DRAPHEAL,515.127.4561,2002-12-07,PU_MAN,11000,-,100,30,Den Popp,OLD Employee
107,Diana,Lorentz,DLORENTZ,590.423.5567,2007-02-07,IT_PROG,4200,-,103,60,Diana Lorentz,OLD Employee
198,Donald,OConnell,DOCONNEL,650.507.9833,2007-06-21,SH_CLERK,2600,-,124,50,Donald OConnell,OLD Employee


In [0]:
# join (inner, left, right, full) --> df1.join(df2, condition, type_of_join)
# left anti --> Display the records available in left table but not available in right table
final_df = emp_final_res.join(dept_df, emp_final_res.DEPARTMENT_ID == dept_df.ID,'inner')


# write this result to specified path
output_path = '/Volumes/student_demo/standard/output/my_data'
final_df.write.format('csv').option('header','true').save(output_path)
# display(final_df)

In [0]:
databricks customer table to --> oracle table (customer)

day1 -->100  --> push to oracle 100
day2 -->40 --> got push to oracle

In [0]:
count in databricks customer table --> 22000
count in oracle customer table ---> 21990


10 missing reocrds ?????????????? data missing scenario

In [0]:
%sql
select * from databricks_customer t1
anti join oracle_customer t2
on t1.id = t2.id
--10 records

In [0]:
data = [
    ('0001', 'Alice', 50000, 'New York','Alice is a dedicated employee.\nShe works in the finance department.'),

    ('0002', 'Bob', 60000, 'Los Angeles','Bob manages the marketing team.\nHe focuses on digital campaigns.'),

    ('0003', 'Charlie', 55000, 'Chicago','Charlie is a software engineer.\nHe loves working on big data projects.'),

    ('0004', 'David', 62000, 'Houston','David leads the sales team.\nHe travels frequently for client meetings.'),

    ('0005', 'Eva', 58000, 'Philadelphia','Eva is responsible for HR.\nShe organizes team-building events.')
]
columns = ['ID','Name','Salary','City','Description']
df = spark.createDataFrame(data, schema = columns)

# Save dataframe to csv
output_path = '/Volumes/student_demo/standard/output/my_description_data'
df.write.format('csv').option('header','true').save(output_path)
# display(df)

In [0]:
display(df)

ID,Name,Salary,City,Description
0001,Alice,50000,New York,Alice is a dedicated employee. She works in the finance department.
0002,Bob,60000,Los Angeles,Bob manages the marketing team. He focuses on digital campaigns.
0003,Charlie,55000,Chicago,Charlie is a software engineer. He loves working on big data projects.
0004,David,62000,Houston,David leads the sales team. He travels frequently for client meetings.
0005,Eva,58000,Philadelphia,Eva is responsible for HR. She organizes team-building events.


In [0]:
path = '/Volumes/student_demo/standard/output/my_description_data'

#multiLine() --> option to handle fields that contain new line character within the text
df1 = spark.read.format('csv').options(header='true',multiLine='true').load(path)

# remove leading zeros from ID column
res = df1.withColumn('ID', regexp_replace('ID', '^0+', ''))
display(res)


ID,Name,Salary,City,Description
3,Charlie,55000,Chicago,Charlie is a software engineer. He loves working on big data projects.
4,David,62000,Houston,David leads the sales team. He travels frequently for client meetings.
1,Alice,50000,New York,Alice is a dedicated employee. She works in the finance department.
2,Bob,60000,Los Angeles,Bob manages the marketing team. He focuses on digital campaigns.
5,Eva,58000,Philadelphia,Eva is responsible for HR. She organizes team-building events.
